# Indexing Data to Elasticsearch

In [34]:
![ ! -d "venv" ] && python3 -m venv preprocessing-env
!source preprocessing-env/bin/activate

In [47]:
!python3 -m pip install --quiet -r requirements.txt

In [91]:
import os
import csv
from pprint import pprint

In [39]:
FILENAME = "./data/all_bikez_curated.csv"

In [78]:
data, count = [], 0
with open(FILENAME, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        data.append(row)
        count += 1

print(f"{count} rows read from {FILENAME}")
data[125]

38472 rows read from ./data/all_bikez_curated.csv


{'Brand': 'adly',
 'Model': 'noble 125',
 'Year': '2008',
 'Category': 'Scooter',
 'Rating': '',
 'Displacement (ccm)': '124.0',
 'Power (hp)': '8.6',
 'Torque (Nm)': '9.3',
 'Engine cylinder': 'Single cylinder',
 'Engine stroke': ' four-stroke',
 'Gearbox': 'Automatic',
 'Bore (mm)': '52.4',
 'Stroke (mm)': '57.8',
 'Fuel capacity (lts)': '6.0',
 'Fuel system': '',
 'Fuel control': '',
 'Cooling system': 'Air',
 'Transmission type': '',
 'Dry weight (kg)': '90.0',
 'Wheelbase (mm)': '1255.0',
 'Seat height (mm)': '',
 'Front brakes': 'Single disc',
 'Rear brakes': 'Expanding brake (drum brake)',
 'Front tire': '120/70-12 ',
 'Rear tire': '120/70-12 ',
 'Front suspension': 'Grease shock absorber',
 'Rear suspension': 'Hydraulic shock absorber',
 'Color options': ''}

In [90]:
# Helper functions for data cleaning

def clean_str(value):
    if value is None:
        return None

    value = value.strip()

    return value if value else None


def clean_int(value):
    value = clean_str(value)

    if value is None:
        return None

    try:
        return int(float(value))
    except ValueError:
        return None


def clean_float(value):
    value = clean_str(value)

    if value is None:
        return None

    try:
        return float(value)
    except ValueError:
        return None

In [67]:
from dotenv import load_dotenv
load_dotenv()

True

In [89]:
ELASTICSEARCH_ENDPOINT = os.getenv("ES_ENDPOINT")
ELASTICSEARCH_API_KEY = os.getenv("ES_API_KEY")

INDEX_NAME = "motorcycles"

In [85]:
from elasticsearch import Elasticsearch, helpers

In [86]:
es = Elasticsearch(
    ELASTICSEARCH_ENDPOINT,
    api_key=ELASTICSEARCH_API_KEY
)

pprint(es.info())

ObjectApiResponse({'name': 'serverless', 'cluster_name': 'a7edf7e824f54dbc9667072e873e9633', 'cluster_uuid': '_nseSM0XSpi0OlakCZ1uQw', 'version': {'number': '9.5.0', 'build_flavor': 'serverless', 'build_type': 'docker', 'build_hash': '2c8993dcd18d62a6f71d023d797a25db1b0884c9', 'build_date': '2026-05-11T15:00:41.796718359Z', 'build_snapshot': False, 'lucene_version': '10.4.0', 'minimum_wire_compatibility_version': '9.5.0', 'minimum_index_compatibility_version': '9.5.0'}, 'tagline': 'You Know, for Search'})


In [92]:
es.indices.delete(index=INDEX_NAME, ignore_unavailable=True)

ObjectApiResponse({'acknowledged': True})

In [93]:
actions = []
for item in data:
    cleaned_item = {
        "brand": clean_str(item.get("Brand")),
        "model": clean_str(item.get("Model")),
        "year": clean_int(item.get("Year")),
        "category": clean_str(item.get("Category")),
        "rating": clean_float(item.get("Rating")),
        "displacement_ccm": clean_float(item.get("Displacement (ccm)")),
        "power_hp": clean_float(item.get("Power (hp)")),
        "torque_nm": clean_float(item.get("Torque (Nm)")),
        "engine_cylinder": clean_str(item.get("Engine cylinder")),
        "engine_stroke": clean_str(item.get("Engine stroke")),
        "gearbox": clean_str(item.get("Gearbox")),
        "bore_mm": clean_float(item.get("Bore (mm)")),
        "stroke_mm": clean_float(item.get("Stroke (mm)")),
        "fuel_capacity_lts": clean_float(item.get("Fuel capacity (lts)")),
        "fuel_system": clean_str(item.get("Fuel system")),
        "fuel_control": clean_str(item.get("Fuel control")),
        "cooling_system": clean_str(item.get("Cooling system")),
        "transmission_type": clean_str(item.get("Transmission type")),
        "dry_weight_kg": clean_float(item.get("Dry weight (kg)")),
        "wheelbase_mm": clean_float(item.get("Wheelbase (mm)")),
        "seat_height_mm": clean_float(item.get("Seat height (mm)")),
        "front_brakes": clean_str(item.get("Front brakes")),
        "rear_brakes": clean_str(item.get("Rear brakes")),
        "front_tire": clean_str(item.get("Front tire")),
        "rear_tire": clean_str(item.get("Rear tire")),
        "front_suspension": clean_str(item.get("Front suspension")),
        "rear_suspension": clean_str(item.get("Rear suspension")),
        "color_options": clean_str(item.get("Color options"))
    }
    cleaned_item = {
        key: value
        for key, value in cleaned_item.items()
        if value is not None
    }
    action = {
        "_index": INDEX_NAME,
        "_source": cleaned_item
    }
    actions.append(action)

In [94]:
helpers.bulk(es, actions)

(38472, [])

Voila! Indexing is completed! We can check the documents on Kibana with the below query:

```sql
GET /motorcycles/_search
{
    "query": {
        "match_all": {}
    }
}
```